# Lekcja 07 - Wzorzec projektowy Planowania

Ten notatnik demonstruje **wzorzec projektowy Planowania** dla agentów AI z wykorzystaniem Microsoft Agent Framework.
Nauczysz się, jak rozbić złożone zapytania podróżne na uporządkowane podzadania, przypisać je do specjalistycznych agentów,
i wykonać powstały plan — wszystko za pomocą uporządkowanego wyjścia opartego na modelach Pydantic.


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Rozkład zadania

Rozkład zadania jest podstawą wzorca projektowego planowania. Zamiast prosić pojedynczego agenta o
obsługę złożonego zapytania od początku do końca, dzielimy problem na mniejsze, dobrze zdefiniowane **podzadania**.
Każde podzadanie jest przypisane do wyspecjalizowanego agenta (np. loty, hotele, atrakcje) z jasnymi
priorytetami i uporządkowaniem zależności.

Takie podejście przynosi kilka korzyści:
- **Jasność**: każde podzadanie ma jedną odpowiedzialność.
- **Równoległość**: niezależne podzadania mogą być wykonywane jednocześnie.
- **Niezawodność**: awarie są izolowane do pojedynczych podzadań.
- **Śledzenie budżetu**: koszty są szacowane dla każdego podzadania i sumowane.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Tworzenie agenta planującego ze strukturalnym wynikiem

Agent planujący działa jako **koordynator recepcji**. Na podstawie wysokopoziomowego zapytania podróżnego
tworzy strukturalny `TravelPlan` — rozkładając zapytanie na podzadania, ustalając priorytety
i identyfikując zależności, tak aby konsjerż lub warstwa wykonawcza mogła zrealizować zadanie.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Realizacja planu za pomocą specjalistycznych narzędzi

Gdy agent recepcji sporządzi uporządkowany plan, realizuje go **agent konsjerża**.
Każde specjalistyczne narzędzie obsługuje jedną kategorię podzadań (loty, hotele, atrakcje). Konsjerż
iteruje przez podzadania planu w kolejności zależności i przekazuje każde z nich do
odpowiedniego narzędzia.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Podsumowanie

W tej lekcji nauczyłeś się **Wzoru Projektowego Planowania** dla agentów AI:

1. **Decompozycja Zadania** — Agent planujący przy recepcji dzieli złożone zapytanie podróżne na
   uporządkowane podzadania, wykorzystując modele Pydantic, przypisując każde do specjalistycznego agenta z priorytetami
   i zależnościami.
2. **Strukturalny Wynik** — Przekazując `response_format` agent zwraca zweryfikowany
   obiekt `TravelPlan` zamiast tekstu w dowolnej formie, co sprawia, że dalsze przetwarzanie jest niezawodne.
3. **Wykonanie Planu** — Agent konsjerża przechodzi przez podzadania, korzystając ze specjalistycznych narzędzi
   (`book_flight`, `reserve_hotel`, `book_activity`) by zrealizować plan i raportować wyniki.

Ten wzorzec oddziela *co robić* (planowanie) od *jak to zrobić* (wykonanie), czyniąc agentów
bardziej modułowymi, testowalnymi i łatwiejszymi do rozbudowy.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Choć dążymy do dokładności, prosimy pamiętać, że automatyczne tłumaczenia mogą zawierać błędy lub niedokładności. Oryginalny dokument w jego języku źródłowym należy uznawać za autorytatywne źródło. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
